# 36 - Humanoid Hands And Tactile Manipulation

## What / Why / How

**What we are trying to do:** Model dexterous hands, tactile feedback, grasp force, slip detection, and manipulation state machines.

**Why this matters:** Optimus-like usefulness depends heavily on hands. Walking matters, but value comes from manipulating the world.

**How we will do it:** Simulate grip force, tactile slip signals, grasp adjustment, and a small manipulation controller.

## Hands And Tactile Feedback

Humanoid usefulness depends on picking up, holding, placing, and using objects. Tactile feedback helps detect slip and contact state.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(36)
object_weight = 1.2
mu = 0.6
fingers = 4
required_normal_force = object_weight * 9.81 / (mu * fingers)
print("minimum per-finger normal force [N]:", round(required_normal_force, 2))

In [ ]:
grip_force = 1.0
log = []
for t in range(60):
    tangential = object_weight * 9.81 / fingers + rng.normal(0, 0.15)
    max_static = mu * grip_force
    slip = tangential > max_static
    if slip:
        grip_force += 0.25
    else:
        grip_force *= 0.998
    log.append((t, grip_force, tangential, max_static, slip))
log = np.array(log, dtype=float)
print("final grip force:", log[-1, 1])
print("slip events:", int(log[:, 4].sum()))

In [ ]:
states = ["approach", "contact", "load_test", "lift", "transport", "place", "release"]
state = 0
events = []
for tick in range(20):
    tactile_contact = tick >= 3
    stable_grip = tick >= 7
    at_place = tick >= 15
    current = states[state]
    events.append((tick, current))
    if current == "approach" and tactile_contact:
        state += 1
    elif current == "contact":
        state += 1
    elif current == "load_test" and stable_grip:
        state += 1
    elif current == "lift":
        state += 1
    elif current == "transport" and at_place:
        state += 1
    elif current == "place":
        state += 1
print(events)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(log[:, 0], log[:, 1], label="grip force")
    plt.plot(log[:, 0], log[:, 2], label="tangential load")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title("Slip-reactive grip")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add fragile-object max force.
2. Add two tactile sensors per finger.
3. Explain why vision alone is not enough for dexterous manipulation.